# Дослідницька програма для базлайну 91.13% (run_20260630_235356)

Ноутбук відповідає на запити керівника: **(1)** наочний post-mortem помилок 2→7;
**(2)** шість досліджень — інші датасети MNIST, залежність концепту від порядку/кількості/аугментації
навчальних прикладів, кроки редукції до сталого концепту, аналіз параметрів концептів.

**Як користуватися**
- Kernel: **natural-agi** (venv проєкту). Якщо його немає в списку — виконайте один раз у терміналі:
  `natural-agi/bin/python -m ipykernel install --user --name natural-agi --display-name "natural-agi"`
- Вся логіка — у пакеті `src/training/supervisor_experiments/`; клітинки лише викликають функції та показують результат.
- 🔴 — руйнівні клітинки (зупиняють/чистять Neo4j). Вони виконуються **тільки** якщо ви явно поставите `CONFIRM = "yes"`, і перевіряють наявність бекапу.
- ⚠️ Пастка Jupyter: відредагована клітинка **не** виконується сама — після зміни коду клітинку треба запустити повторно.
- Довгі клітинки друкують час виконання і зберігають артефакти на диск (повторний запуск безпечний).

**Секції:** 0 — налаштування · 1 — post-mortem 2→7 · 2 — параметри концептів (S6) · 3 — збіжність редукції (S5) ·
4 — порядок пред'явлення (S2) · 5 — кількість прикладів (S3) · 6 — датасети MNIST (S1) · 7 — 🔴 ретрени (S3/S4) · 8 — зведення.

In [ ]:
import logging, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _c in (_here, *_here.parents):
    if (_c / "src" / "training" / "supervisor_experiments").exists():
        REPO = _c
        sys.path.insert(0, str(_c / "src" / "training"))
        break
else:
    raise RuntimeError("Запустіть Jupyter з кореня репозиторію або з src/training/")

logging.disable(logging.INFO)  # приглушити DEBUG/INFO продукційних модулів
%matplotlib inline

from supervisor_experiments import infra, report
from supervisor_experiments import postmortem as pm

RUN_ID = pm.RUN_ID
print(f"REPO = {REPO}")
print(f"Базлайн: {RUN_ID} (91.13%)")

### 0.1 Перевірка середовища

Очікування: Neo4j запущений і містить **13 концептів** та **8 685 графів зображень** базлайн-прогону
(`baseline_state_intact: True`). Для Секцій 1–5 достатньо самого Neo4j; Kafka/Nuclio потрібні лише
для Секцій 6–7 (`make start_services` + `docker start $(docker ps -aq --filter name=nuclio-nuclio)`).

In [ ]:
health = infra.health_check()
for k, v in health.items():
    print(f"  {k}: {v}")
assert health["neo4j_container_up"], "Neo4j не запущений: docker start naturalagi-neo4j-1"
assert health["baseline_state_intact"], "Стан Neo4j НЕ відповідає базлайну — див. Секцію 7 (відновлення з бекапа)"
print("\n✅ Середовище готове (Tier A)")

### 0.2 🔴 Бекап Neo4j (обов'язковий шлюз перед Секцією 7)

Холодний бекап volume: Neo4j зупиняється на ~1 хв, у `backups/neo4j_baseline_2026-07-17.tar.gz`
зберігається весь стан (концепти + 8 685 графів зображень). Без цього файлу руйнівні клітинки Секції 7 відмовляться працювати.

In [ ]:
CONFIRM = "no"  # ← поставте "yes", щоб зробити бекап (Neo4j зупиниться на ~1 хв)

if CONFIRM == "yes":
    infra.backup_neo4j(CONFIRM)
else:
    print(f"Бекап існує: {infra.BACKUP_PATH.exists()} ({infra.BACKUP_PATH})")
    print('Щоб створити/оновити: CONFIRM = "yes" і перезапустіть клітинку.')

## Секція 1 — Post-mortem: чому «2» падають у «7»

**Механізм (підтверджено на всіх 111 промахах 2→7):** двоступеневий, зі спільною першопричиною —
втратою якірних точок при побудові графа (ще до редукції):

1. Пре-фільтр складності (`classification_orchestrator.py:97-102`) відкидає концепти зі
   `складність > складність_зображення` **до** порівняння. Збіднений скелет «2» має медіанну складність 21,
   а «повний» концепт 2_2 — 24 → у **77.5%** випадків 2_2 взагалі не бере участі у WTA-конкуренції.
2. Спрощений 2_1 (складність 13) конкурує, але програє 7_1 за сирою схожістю (середній розрив 0.14):
   вцілілі точки стоять «не там» (OUT-of-range за просторовими ознаками).

Кожна клітинка нижче відтворює один крок аналізу продукційним кодом. Підсумковий документ:
[`researches/two_to_seven_postmortem.md`](../../researches/two_to_seven_postmortem.md).

In [ ]:
params = pm.load_params()
concepts = pm.load_run_concepts()
confusion = pm.load_confusion(expected="2", predicted="7")
print(f"Концептів у снепшоті прогону: {len(concepts)}")
print(f"Промахів 2→7 у {RUN_ID}: {len(confusion)}")
confusion.head(3)

**1.1 Як виглядають ці «2»** (перші 12 із 111):

In [ ]:
fig = pm.show_images(confusion, n=12)
report.save_figure(fig, "two_to_seven/misclassified_2_examples.png")

**1.2 Продукційний ранкінг еталонного прикладу** — `mnist_test_2_00766` (канонічна «2» із петлею).
Читання виводу: `raw_sim` — сира GED-схожість; `adjusted = raw + 0.10·log₂(складність)`;
концепт **відсутній у списку** ⇔ його зрізав пре-фільтр складності (тут: 2_2 і 8_1).
Нижче — per-feature розбір програшу 2_1 (OUT-of-range ⇒ повний NO_MATCH за Family E).

In [ ]:
print(pm.explain_cli(pm.PRIMARY_EXEMPLAR, expected="2"))

**1.3 Бакет-аналіз усіх 111 промахів.** Бакети: **A** — жоден 2-концепт не пройшов пре-фільтр;
**B** — конкурував, програв за сирою схожістю; **C** — виграв raw, програв λ-ранжування.
Додатково рахуємо, чи був відфільтрований саме 2_2 (`c22_prefiltered`).

In [ ]:
buckets = pm.bucket_all(list(confusion.image_id), concepts, params, expected="2")
buckets = buckets.merge(confusion[["image_id", "image_path"]], on="image_id")
buckets.to_csv(report.FIGURES / "two_to_seven" / "bucket_analysis.csv", index=False)
display(pm.bucket_summary(buckets))
n22 = int(buckets.c22_prefiltered.sum())
print(f"2_2 відфільтровано пре-фільтром: {n22}/{len(buckets)} ({n22/len(buckets)*100:.1f}%)")
print(f"Збіг прогнозу інструмента з прогоном: "
      f"{int((buckets.predicted_tool == '7_1').sum())}/{len(buckets)}")
fig = pm.plot_buckets(buckets)
report.save_figure(fig, "two_to_seven/bucket_bar.png")

**1.4 Триптих для керівника:** зображення → граф до редукції → концепти 2_2 (відфільтровано) і 7_1 (переможець).

In [ ]:
row = confusion[confusion.image_id == pm.PRIMARY_EXEMPLAR].iloc[0]
fig = pm.triptych(pm.PRIMARY_EXEMPLAR, row, concepts, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00766_triptych.png")

**1.5 Стадії побудови графа** — де саме зникають якірні точки: в оригіналі голова «2» — замкнена петля;
після бінаризації + 1px-thinning вона стає відкритим гачком (0 циклів), GNG+RDP сплющують дугу в 2 сегменти.

In [ ]:
fig = pm.construction_figure(row, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00766_construction.png")

**1.6 Контрастний приклад** — `mnist_test_2_00984`: пласка курсивна «2», «чесно» схожа на 7 (не всі промахи — жертви скелетонізації).

In [ ]:
row_c = confusion[confusion.image_id == pm.CONTRAST_EXEMPLAR].iloc[0]
fig = pm.triptych(pm.CONTRAST_EXEMPLAR, row_c, concepts, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00984_triptych.png")
fig = pm.construction_figure(row_c, params)
report.save_figure(fig, "two_to_seven/mnist_test_2_00984_construction.png")

### Висновок (Секція 1)

- Складніший концепт «2» не програє WTA-конкуренцію — **він у неї не потрапляє**: пре-фільтр складності
  зрізає 2_2 у 86/111 випадків (77.5%), бо збіднений скелет має складність ≤ 21 < 24.
- Причина збіднення видима на фігурі стадій: **петля голови «2» руйнується на бінаризації/thinning**
  (0 циклів у скелеті), GNG+RDP довершують спрощення. Це і є «втрата якірних точок до редукції».
- Навіть уцілілий 2_1 програє 7_1 за сирою схожістю (розрив 0.14) — вцілілі точки геометрично зміщені
  (OUT-of-range за `distance_to_centroid`, `normalized_x`).
- λ-ранжування невинне: бакет C = 0. Головний важіль — **скелетонізація, що зберігає петлі**;
  пом'якшення пре-фільтра — лише симптоматичне (властивісне, без хардкоду concept_id).

Повний текст: `researches/two_to_seven_postmortem.md`.

## Секція 2 — Параметри концептів і компресія (S6)

### 2.0 Підготовка даних формування (спільна для Секцій 2–5)

Після успішного створення концепту сервіс **видаляє** session-графи навчальних зразків із Neo4j
(`critical_point_concept_service.py:168-172`), тож їх треба одноразово ре-інгестувати через пайплайн
(додатково, неруйнівно: ~875 зображень ≈ кілька хвилин). Потрібен живий пайплайн
(`make start_services` + Nuclio-функції; див. Секцію 0). Далі графи експортуються у JSON для
офлайн-проб — після цього Секції 2–5 працюють без Kafka/Nuclio.

⚠️ Навчальні вибірки **вже аугментовані**: файли `mnist_2_00006_aug1..aug10` + оригінал без суфікса.

In [ ]:
from supervisor_experiments import formation_lab as fl
import pandas as pd
status = pd.DataFrame(fl.sessions_status()).T
display(status)
need_ingest = (status.graphs_in_neo4j < status.files_on_disk).any()
print("Треба інгестувати" if need_ingest else "✅ Session-графи вже у Neo4j")

In [ ]:
# Одноразово: ре-інгест відсутніх session-графів (потрібен живий пайплайн!)
fl.ingest_missing_sessions()

In [ ]:
# Експорт session-графів у JSON (основа офлайн-проб Секцій 2–5)
for cid in fl.ALL_CONCEPTS:
    samples = fl.SAMPLES_ROOT / cid
    if not samples.exists() or not any(samples.glob("*.json")):
        fl.export_session_graphs(cid, samples)
print("✅ Експортовано")

### 2.1 Які параметри зберігають концепти і наскільки стискається їх первинна кількість

Конвенція підрахунку: range-властивість `{min, max, center}` = 2 збережені числа (center — похідний);
скаляр/рядок = 1; список = довжина; labels = кількість міток; службові властивості (id/session) не рахуються.

In [ ]:
from supervisor_experiments import parameter_compression as pc, postmortem as pm, report
snapshot = pm.load_run_concepts()
table = pc.compression_table(snapshot)
display(table)
report.write_text("concept_parameter_compression.md",
    "# S6: Параметри концептів і компресія\n\n" + report.df_to_markdown(table, floatfmt=".1f") + "\n")

In [ ]:
survival = pc.property_survival(snapshot)
display(survival)
kinds = pc.property_kinds(snapshot)
display(kinds.head(10))
with open(report.RESEARCHES / "concept_parameter_compression.md", "a") as f:
    f.write("\n## Виживання властивостей (частка вузлів, що несуть властивість)\n\n"
            + report.df_to_markdown(survival, floatfmt=".3f")
            + "\n\n## Типи властивостей за видом вузла\n\n"
            + report.df_to_markdown(kinds, floatfmt=".2f") + "\n")

### Висновок (Секція 2)

Заповніть після запуску: скільки range-конвертів несе типовий вузол концепту (~27–29), сумарна
топологічна компресія (Σ вузлів входу / вузлів концепту) і компресія значень; які властивості
не виживають злиття (рядкові, що відрізняються між зразками → None; списки → перетин).

## Секція 3 — За скільки кроків редукції отримуємо сталий концепт (S5)

Два рівні відповіді: **внутрішній** — скільки ітерацій критично-точкової редукції
(endpoint → intersection → corner, ліміт 6, збіжність = ізоморфізм критичних графів) потребує кожне
злиття; **зовнішній** — після скількох зразків концепт перестає змінюватися (фікспойнт топології +
насичення конвертів). Проби виконуються офлайн продукційним кодом (~15–40 хв на всі 13 концептів).

In [ ]:
from supervisor_experiments import convergence as cv
conv_df = cv.run_convergence_probes()
display(conv_df)

In [ ]:
fig = cv.plot_trajectories()
report.save_figure(fig, "convergence/trajectories.png")

In [ ]:
stab = cv.stability_table()
display(stab)
fig = cv.plot_cpp_histogram()
report.save_figure(fig, "convergence/cpp_iterations_hist.png")
report.write_text("concept_convergence_findings.md",
    "# S5: Збіжність редукції\n\n## Сталість концепту (номер зразка m)\n\n"
    + report.df_to_markdown(stab, floatfmt=".0f")
    + "\n\n![Траєкторії](figures/convergence/trajectories.png)\n"
    + "![Ітерації](figures/convergence/cpp_iterations_hist.png)\n")

### Висновок (Секція 3)

Заповніть: типове m_topology (коли фіксується топологія) проти m_envelope (коли насичуються
діапазони) та розподіл внутрішніх ітерацій (скільки злиттів обходяться 0–1 ітерацією, скільки
впираються в ліміт 6 — маркер топологічної гетерогенності, як у 8_1).

## Секція 4 — Чи залежить концепт від послідовності пред'явлення (S2)

Відповідь на рівні коду — **так**: перший зразок стає зародком концепту
(`critical_point_concept_service.py:87-88`), а історично порядок вибірки з Neo4j був
недетермінованим (запит без `ORDER BY` + паралельний Kafka-інгест) — виправлено в цій гілці
(`ORDER BY image_id`). Тут вимірюємо, наскільки сильно порядок змінює результат: 10 сідів × 3
концепти (2_2 — середній, 7_1 — найменший, 8_1 — схильний до крешів на гетерогенній топології).

In [ ]:
from supervisor_experiments import order_dependence as od
order_df = od.run_order_probes()
display(order_df)

In [ ]:
ged_matrices = {}
for cid in od.DEFAULT_CONCEPTS:
    ged_matrices[cid] = od.pairwise_ged(cid)
    print(f"— {cid}:")
    display(ged_matrices[cid])
summary = od.summarize(order_df, ged_matrices)
display(summary)
report.write_text("order_dependence_findings.md",
    "# S2: Залежність від порядку пред'явлення\n\n"
    "Кодовий факт: перший зразок = зародок; порядок з Neo4j був недетермінований до ORDER BY-фіксу.\n\n"
    + report.df_to_markdown(summary, floatfmt=".2f") + "\n")

### Висновок (Секція 4)

Заповніть: частка пар сідів з ідентичною топологією, середній/максимальний попарний GED,
частота крешів формування (особливо 8_1). Якщо розкид ненульовий — рекомендація: канонічний
порядок (вже увімкнений ORDER BY) або відбір найкращого сіда за офлайн-метрикою ширини конвертів.

## Секція 5 — Як залежить побудова концепту від кількості прикладів (S3, офлайн)

Формування на підвибірках N ∈ {2, 5, 10, 20} × 3 сіди для всіх 13 концептів (~156 проб, ~30–60 хв;
звузьте `CONCEPTS_S3`, якщо треба швидше). ⚠️ Файли вибірки корельовані (10 аугментацій на джерело),
тож «ефективна» кількість незалежних прикладів ≈ N/10 — врахуйте в інтерпретації.
Downstream-точність для обраних N — у Секції 7 (потрібен ретрен).

In [ ]:
from supervisor_experiments import sample_count as sc
CONCEPTS_S3 = sc.ALL_CONCEPTS  # звузьте за потреби, напр. ["2_2", "7_1", "8_1", "3_1"]
sc_df = sc.run_sample_count_grid(CONCEPTS_S3)
display(sc.aggregate(sc_df))

In [ ]:
fig = sc.plot_curves(sc_df, full_reference=conv_df if "conv_df" in dir() else None)
report.save_figure(fig, "sample_count/curves.png")
display(sc.failure_table(sc_df))
report.write_text("sample_count_findings.md",
    "# S3: Залежність від кількості прикладів (офлайн-частина)\n\n"
    + report.df_to_markdown(sc.aggregate(sc_df), floatfmt=".2f")
    + "\n\n![Криві](figures/sample_count/curves.png)\n"
    + "\n\n## Частка крешів формування, %\n\n"
    + report.df_to_markdown(sc.failure_table(sc_df), floatfmt=".0f")
    + "\n\n(Downstream-точність — див. Секцію 7 і augmentation_findings.md)\n")

### Висновок (Секція 5)

Заповніть: чи стабілізується топологія вже на малих N (порівняйте з m_topology Секції 3), як росте
ширина конвертів зі збільшенням N (ризик catch-all), чи є N, після якого додаткові зразки лише
розширюють діапазони без зміни структури.

## Секція 6 — Інші датасети MNIST (S1)

Потрібен живий пайплайн. Прогони (базлайн-параметри, `fraction=1.0`):
1. **10k тест без виключень** (~1–2 год) — додає ~1 296 incomplete-зображень;
2. **60k train-спліт**: без виключень і complete-only (~7–14 год кожен, `delete_image_nodes=true`);
3. **70k** = агрегація прогонів 1+2 (окремий прогін не потрібен).

Розмітка повноти для 60k — евристична (`scripts/completeness_heuristic.py`, без ручної перевірки —
зазначайте як caveat). Каталог `datasets/mnist_train_all/` будується один раз (~10–20 хв).

In [ ]:
from supervisor_experiments import dataset_variants as dv
display(dv.manifest_counts(dv.TEST_MANIFEST))

In [ ]:
# Побудова 60k-каталогу (одноразово, ~10–20 хв; idempotent)
import subprocess, sys
if not dv.TRAIN_MANIFEST.exists() or sum(1 for _ in open(dv.TRAIN_MANIFEST)) < 60000:
    proc = subprocess.run([sys.executable, str(dv.REPO / "scripts" / "build_mnist_train_dataset.py")],
                          cwd=dv.REPO, text=True)
    print("exit:", proc.returncode)
display(dv.manifest_counts(dv.TRAIN_MANIFEST))

In [ ]:
for exp in dv.s1_experiments():
    dv.ensure_experiment(exp)

In [ ]:
# ДОВГА КЛІТИНКА (~1–2 год): 10k тест без виключень
dv.run_experiment("exp_s1_test10k_unfiltered", fraction=1.0)

In [ ]:
run10k = dv.load_queue()
exp = next(e for e in run10k["experiments"] if e["id"] == "exp_s1_test10k_unfiltered")
RUN_10K = exp["quick_run_id"] if exp["full_run_id"] is None else exp["full_run_id"]
split10k = dv.split_metrics(RUN_10K, dv.TEST_MANIFEST)
display(split10k)  # порівняйте complete-рядок із базлайном 91.13%

In [ ]:
# ДУЖЕ ДОВГІ КЛІТИНКИ (~7–14 год кожна): 60k train-спліт
dv.run_experiment("exp_s1_train60k_unfiltered", fraction=1.0)

In [ ]:
dv.run_experiment("exp_s1_train60k_complete", fraction=1.0)

In [ ]:
q = dv.load_queue()
def _run_of(eid):
    e = next(x for x in q["experiments"] if x["id"] == eid)
    return e["full_run_id"] or e["quick_run_id"]
RUN_60K_ALL = _run_of("exp_s1_train60k_unfiltered")
rows = [("10k тест", RUN_10K, dv.TEST_MANIFEST), ("60k train", RUN_60K_ALL, dv.TRAIN_MANIFEST)]
frames = []
for label, rid, manifest in rows:
    df = dv.split_metrics(rid, manifest); df["dataset"] = label; frames.append(df)
import pandas as pd
split_all = pd.concat(frames, ignore_index=True)
agg70k = dv.aggregate_runs([(RUN_10K, dv.TEST_MANIFEST), (RUN_60K_ALL, dv.TRAIN_MANIFEST)])
display(split_all); display(agg70k.tail(3))
report.write_text("dataset_variants_findings.md",
    "# S1: Класифікація на інших датасетах MNIST\n\n"
    "Якір: complete-only 10k = 91.13% (run_20260630_235356).\n\n"
    + report.df_to_markdown(split_all, floatfmt=".2f")
    + "\n\n## 70k (агрегація 10k+60k)\n\n"
    + report.df_to_markdown(agg70k.tail(3), floatfmt=".2f")
    + "\n\nCaveats: 60k розмічено евристикою; ~875 навчальних зразків походять із train-спліту (leakage ≤1.5%).\n")

### Висновок (Секція 6)

Заповніть: наскільки падає точність на incomplete-підмножині (очікувано різко: розірвані контури →
збіднені графи → малі концепти-атрактори + DLQ), чи тримається complete-точність на 60k
(генералізація на новий спліт), підсумкові числа для 10k/60k/70k у обох режимах.

## Секція 7 — 🔴 Ретрени: кількість прикладів downstream (S3) і аугментація (S4)

**Руйнівно**: кожен ретрен замінює концепти в Neo4j. Правила: (1) бекап із Секції 0 обов'язковий;
(2) після серії умов — відновлення базлайну; (3) QUICK 25% для сканів, повний прогін лише для
фіналістів (`baseline_accuracy` в queue.json — тільки з повних прогонів).

Вибірки вже аугментовані (~10 варіацій + оригінал на джерело), тому умови S4 будуються
підмножинами наявних файлів: `aug0` = лише оригінали (без аугментації), `aug3` = оригінал+3,
`baseline` = всі, `aug2x` = всі + нові варіації (`scripts/augment_train.py`: поворот ±12°,
масштаб 0.85–1.15, еластика, товщина штриха).

In [ ]:
from supervisor_experiments import tier_c
tier_c.ensure_backup()
baseline_nodes = tier_c.concept_node_counts()
print("Базлайн-вузли:", baseline_nodes)

In [ ]:
# Умовні набори (копії у datasets/train_subsets/<умова>/)
tier_c.build_condition_dirs("N5_seed0", tier_c.subset_n(5, 0))
tier_c.build_condition_dirs("N20_seed0", tier_c.subset_n(20, 0))
tier_c.build_condition_dirs("aug0", tier_c.aug_cap(0))
tier_c.build_condition_dirs("aug3", tier_c.aug_cap(3))

In [ ]:
# Нові аугментації поверх наявних (aug2x)
import subprocess, sys
subprocess.run([sys.executable, str(tier_c.REPO / "scripts" / "augment_train.py"),
                "--dst", str(tier_c.SUBSETS_ROOT / "aug2x"), "--per-image", "1"],
               cwd=tier_c.REPO, check=True)

In [ ]:
# 🔴 РЕТРЕН ОДНІЄЇ УМОВИ + QUICK-оцінка. Повторіть для кожної умови по черзі.
CONDITION = "aug0"          # ← N5_seed0 | N20_seed0 | aug0 | aug3 | aug2x
CONFIRM = "no"              # ← "yes" щоб виконати ретрен

if CONFIRM == "yes":
    result = tier_c.retrain_from(tier_c.SUBSETS_ROOT / CONDITION, confirm=CONFIRM)
    display(result)
    print("Δ вузлів vs базлайн:",
          {c: tier_c.concept_node_counts().get(c, 0) - n for c, n in baseline_nodes.items()})
    from supervisor_experiments import dataset_variants as dv
    exp_id = f"exp_s34_{CONDITION}"
    dv.ensure_experiment({
        "id": exp_id, "name": f"s34_{CONDITION}",
        "description": f"S3/S4 умова {CONDITION} (QUICK 25%)",
        "classes": list(range(10)), "params": dv.baseline_params(),
    })
    dv.run_experiment(exp_id, fraction=0.25)
else:
    print('Поставте CONFIRM = "yes" і оберіть CONDITION.')

In [ ]:
# 🔴 ВІДНОВЛЕННЯ БАЗЛАЙНУ після серії умов
CONFIRM_RESTORE = "no"      # ← "yes" щоб відновити

if CONFIRM_RESTORE == "yes":
    tier_c.restore_baseline(CONFIRM_RESTORE)
    from supervisor_experiments import infra
    print(infra.health_check())
else:
    print('Поставте CONFIRM_RESTORE = "yes".')

In [ ]:
# Зведення умов S3/S4 (після QUICK-прогонів)
q = dv.load_queue()
rows = [{"умова": e["id"].replace("exp_s34_", ""), "quick_accuracy": e["quick_accuracy"]}
        for e in q["experiments"] if e["id"].startswith("exp_s34_")]
import pandas as pd
s34 = pd.DataFrame(rows)
display(s34)
report.write_text("augmentation_findings.md",
    "# S3 (downstream) / S4: ретрени за умовами\n\n"
    "QUICK 25% — лише для порівняння між умовами (не базлайн).\n\n"
    + report.df_to_markdown(s34, floatfmt=".2f") + "\n")

### Висновок (Секція 7)

Заповніть: як точність залежить від N (5/20/всі) і від фактора аугментації (0/3/10/10+нові);
частота крешів формування за умовами (гетерогенність топології зростає з аугментацією);
чи виправдана поточна 10× аугментація проти помірної.

## Секція 8 — Зведений звіт для зустрічі

In [ ]:
report.write_program_summary()
print(open(report.RESEARCHES / "supervisor_program_summary.md").read())